# NetFlow v1 Preprocess

NF-CSE-CIC-IDS2018 데이터셋을 기존 학습 산출물과 분리된 `processed_netflow_v1` 버전으로 전처리한다.

In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'training').exists():
    PROJECT_ROOT = Path.cwd().parents[1]

BASE_FEATURES = [
    'L4_SRC_PORT',
    'L4_DST_PORT',
    'PROTOCOL',
    'L7_PROTO',
    'IN_BYTES',
    'OUT_BYTES',
    'IN_PKTS',
    'OUT_PKTS',
    'TCP_FLAGS',
    'FLOW_DURATION_MILLISECONDS',
]
DERIVED_FEATURES = [
    'TOTAL_BYTES',
    'TOTAL_PKTS',
    'BYTES_PER_PACKET',
    'BYTES_PER_SECOND',
    'PACKETS_PER_SECOND',
    'IN_OUT_BYTES_RATIO',
    'IN_OUT_PKTS_RATIO',
]
FEATURE_NAMES = [*BASE_FEATURES, *DERIVED_FEATURES]
LABEL_MAPPING = {
    'Benign': 0,
    'Bot': 1,
    'Brute Force': 2,
    'DDoS': 3,
    'DoS': 4,
    'Infiltration': 5,
    'SQL Injection': 6,
}

def normalize_attack(value: str) -> str:
    attack = str(value).strip()
    lowered = attack.lower()
    if lowered == 'benign':
        return 'Benign'
    if 'bot' in lowered:
        return 'Bot'
    if 'brute' in lowered or lowered in {'ftp-bruteforce', 'ssh-bruteforce'}:
        return 'Brute Force'
    if 'ddos' in lowered:
        return 'DDoS'
    if lowered.startswith('dos'):
        return 'DoS'
    if 'infilteration' in lowered or 'infiltration' in lowered:
        return 'Infiltration'
    if 'sql injection' in lowered:
        return 'SQL Injection'
    raise ValueError(f'지원하지 않는 Attack 라벨입니다: {value!r}')

def add_derived_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    duration_ms = result['FLOW_DURATION_MILLISECONDS'].clip(lower=1)
    duration_seconds = duration_ms / 1000.0
    result['TOTAL_BYTES'] = result['IN_BYTES'] + result['OUT_BYTES']
    result['TOTAL_PKTS'] = result['IN_PKTS'] + result['OUT_PKTS']
    result['BYTES_PER_PACKET'] = result['TOTAL_BYTES'] / result['TOTAL_PKTS'].replace(0, np.nan)
    result['BYTES_PER_SECOND'] = result['TOTAL_BYTES'] / duration_seconds
    result['PACKETS_PER_SECOND'] = result['TOTAL_PKTS'] / duration_seconds
    result['IN_OUT_BYTES_RATIO'] = result['IN_BYTES'] / result['OUT_BYTES'].replace(0, np.nan)
    result['IN_OUT_PKTS_RATIO'] = result['IN_PKTS'] / result['OUT_PKTS'].replace(0, np.nan)
    result.replace([np.inf, -np.inf], 0.0, inplace=True)
    result.fillna(0.0, inplace=True)
    return result

def sample_by_class(frame: pd.DataFrame, *, max_rows_per_class: int | None, random_state: int) -> pd.DataFrame:
    if max_rows_per_class is None or max_rows_per_class <= 0:
        return frame
    sampled = []
    for _, group in frame.groupby('label_name', sort=False):
        if len(group) > max_rows_per_class:
            sampled.append(group.sample(max_rows_per_class, random_state=random_state))
        else:
            sampled.append(group)
    return pd.concat(sampled, ignore_index=True)

DATASET_PATH = PROJECT_ROOT / 'training/data/88a47ba2ab64258e_MOHANAD_A4706/data/NF-CSE-CIC-IDS2018.csv'
OUTPUT_DIR = PROJECT_ROOT / 'training/data/processed_netflow_v1'
MAX_ROWS_PER_CLASS = 200_000
TEST_SIZE = 0.15
VAL_SIZE = 0.15
RANDOM_STATE = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH

PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/data/88a47ba2ab64258e_MOHANAD_A4706/data/NF-CSE-CIC-IDS2018.csv')

In [2]:
usecols = [*BASE_FEATURES, 'Attack']
raw = pd.read_csv(DATASET_PATH, usecols=usecols)
raw['label_name'] = raw['Attack'].map(normalize_attack)
raw['label'] = raw['label_name'].map(LABEL_MAPPING).astype(np.int64)

print(raw.shape)
raw['label_name'].value_counts()

(8392401, 13)


label_name
Benign           7373198
DDoS              380096
Brute Force       291955
DoS               269361
Infiltration       62072
Bot                15683
SQL Injection         36
Name: count, dtype: int64

In [3]:
sampled = sample_by_class(
    raw,
    max_rows_per_class=MAX_ROWS_PER_CLASS,
    random_state=RANDOM_STATE,
)

feature_frame = add_derived_features(sampled[BASE_FEATURES]).astype(np.float32)
labels = sampled['label'].to_numpy(dtype=np.int64)

print(sampled.shape)
sampled['label_name'].value_counts()

(877791, 13)


label_name
Benign           200000
Brute Force      200000
DDoS             200000
DoS              200000
Infiltration      62072
Bot               15683
SQL Injection        36
Name: count, dtype: int64

In [4]:
x_train_val, x_test, y_train_val, y_test = train_test_split(
    feature_frame[FEATURE_NAMES].to_numpy(dtype=np.float32),
    labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=labels,
)
val_ratio = VAL_SIZE / (1.0 - TEST_SIZE)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_val,
    y_train_val,
    test_size=val_ratio,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train).astype(np.float32)
x_val_scaled = scaler.transform(x_val).astype(np.float32)
x_test_scaled = scaler.transform(x_test).astype(np.float32)

x_train_scaled.shape, x_val_scaled.shape, x_test_scaled.shape

((614453, 17), (131669, 17), (131669, 17))

In [5]:
np.save(OUTPUT_DIR / 'X_train.npy', x_train_scaled)
np.save(OUTPUT_DIR / 'X_val.npy', x_val_scaled)
np.save(OUTPUT_DIR / 'X_test.npy', x_test_scaled)
np.save(OUTPUT_DIR / 'y_train.npy', y_train)
np.save(OUTPUT_DIR / 'y_val.npy', y_val)
np.save(OUTPUT_DIR / 'y_test.npy', y_test)

with (OUTPUT_DIR / 'scaler.pkl').open('wb') as file:
    pickle.dump(scaler, file)

(OUTPUT_DIR / 'feature_names.json').write_text(
    json.dumps(FEATURE_NAMES, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
(OUTPUT_DIR / 'label_mapping.json').write_text(
    json.dumps(LABEL_MAPPING, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

meta = {
    'version': 'netflow_v1',
    'source': str(DATASET_PATH),
    'num_features': len(FEATURE_NAMES),
    'feature_names': FEATURE_NAMES,
    'label_mapping': LABEL_MAPPING,
    'rows': int(len(sampled)),
    'max_rows_per_class': MAX_ROWS_PER_CLASS,
    'class_counts': sampled['label_name'].value_counts().to_dict(),
    'splits': {
        'train': int(len(y_train)),
        'val': int(len(y_val)),
        'test': int(len(y_test)),
    },
}
(OUTPUT_DIR / 'preprocess_meta.json').write_text(
    json.dumps(meta, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

OUTPUT_DIR

PosixPath('/Users/minseok/Desktop/codex/프로젝트/DeepFence/training/data/processed_netflow_v1')